# Useful methods to use Crystal elements from Sage

## Actually... no idea what this is

In [3]:
R = ZZ['t']

t = R.gens()[0]

def give_states(r,n):
    K = crystals.KirillovReshetikhin(['C',n,1],r,1)
    states = []
    for k in K:
        states.append([k.to_tableau()[i][0] for i in range(r)])
    return(states)

def absolute_value(s):
    sor = [abs(x) for x in s]
    sor.sort()
    return(sor)

def Z(r,n):
    states = give_states(r,n)
    bs = states[0]

    ans = 0

    for s in states:
        diffs = [ abs( absolute_value(s)[i] - absolute_value(bs)[i] ) for i in range(r) ]
        ans += t^( sum(diffs) )
    
    return(ans)

## Processing KN columns and the splitting map in type C (a.k.a virtualization)

In [1]:
# Turns the crystal element x into an array of arrays of numbers
def processing_element(x):
    ans = []
    for e in x:
        r = len(e.to_tableau())
        ans.append([e.to_tableau()[i][0] for i in range(r)])
    return(ans)

# Converts a row of a processed element (using processing_element in a crystal tensor product element) into an array of particles
# Convention: [...,[type,label],...] where type is negative if square, positive if circle, and |type| is the position of the particle in a 
#                                          segment of "n" sites, and
#                                          label is acquired by the labelling process later... initializes at 0

def turn_to_particles_row(n,row):
    particles = []
    for r in row:
        particles.append([r,0])
    return(particles)

def turn_to_particles(n,p):
    el = processing_element(p)
    ans = []
    for row in el:
        ans.append(turn_to_particles_row(n,row))
    return(ans)


# Split!

def sort_by_absolute_value(arr):
    return sorted(arr, key=abs)

def split_row(n,row):
    #Below
    r1 = []
    #Above
    r2 = []

    #Looking for particles
    repeats = list(reversed([x for x in range(1,n+1) if (x in row and -x in row)]))
    
    diff = [x for x in row if x not in repeats and -x not in repeats]

    spaces = [x for x in range(1,n+1) if x not in row and -x not in row]
    
    for x in repeats:
        r1.append(-x)
        r2.append(x)
    for y in diff:
        r1.append(y)
        r2.append(y)

    for r in repeats:
        t = max([s for s in spaces if s < r])
        spaces.remove(t)

        r1.append(t)
        r2.append(-t)

    return([sort_by_absolute_value(r1),sort_by_absolute_value(r2)])
    

def split(n,p):
    el = processing_element(p)
    ans = []
    for row in el:
        ans += split_row(n,row)
    return(ans)
  

def to_number_matrix(n,p):
    x = split(n,p)
    ans = []
    for row in x:
        particles = [0 for i in range(n)]
        for j in row:
            if j>0:
                particles[j-1] = 1
            elif j<0:
                particles[-j-1] = -1

        ans.append(particles)
    return(ans)

def split_array(n,p):
    el = p
    ans = []
    for row in el:
        ans += split_row(n,row)
    return(ans)

def to_number_matrix_arrays(n,p):
    x = split_array(n,p)
    ans = []
    for row in x:
        particles = [0 for i in range(n)]
        for j in row:
            if j>0:
                particles[j-1] = 1
            elif j<0:
                particles[-j-1] = -1

        ans.append(particles)
    return(ans)
    

## Computing Type C R-matrix for two rows

In [1]:
r1 = 2
r2 = 3
n = 4

B1 = crystals.KirillovReshetikhin(['C',n,1],r1,1)
B2 = crystals.KirillovReshetikhin(['C',n,1],r2,1)
T = crystals.TensorProduct(B1,B2)

Rmat = B1.R_matrix(B2)

x1 = B1.list()[10]
x2 = B2.list()[10]

x = T([x1,x2])

Rmat(T.list()[10])

[[[1], [2], [-3]], [[1], [4]]]

In [150]:
def process_row(row):
    return [[x] for x in row]

In [156]:
def R_matrix_A_two_rows(n,r1,r2,row1,row2):
    B1 = crystals.KirillovReshetikhin(['A',n,1],r1,1)
    B2 = crystals.KirillovReshetikhin(['A',n,1],r2,1)
    T = crystals.TensorProduct(B1,B2)
    Rmat = B1.R_matrix(B2)
    
    prow1 = process_row(row1)
    prow2 = process_row(row2)

    x1 = B1(rows=prow1)
    x2 = B2(rows=prow2)

    x = T(x1,x2)

    return(Rmat(x))
    
def R_matrix_C_two_rows(n,r1,r2,row1,row2):
    B1 = crystals.KirillovReshetikhin(['C',n,1],r1,1)
    B2 = crystals.KirillovReshetikhin(['C',n,1],r2,1)
    T = crystals.TensorProduct(B1,B2)
    Rmat = B1.R_matrix(B2)
    
    prow1 = process_row(row1)
    prow2 = process_row(row2)

    x1 = B1(rows=prow1)
    x2 = B2(rows=prow2)

    x = T(x1,x2)

    return(Rmat(x))

def R_matrix_A_two_crystal_rows(n,r1,r2,row1,row2):
    B1 = crystals.KirillovReshetikhin(['A',n,1],r1,1)
    B2 = crystals.KirillovReshetikhin(['A',n,1],r2,1)
    T = crystals.TensorProduct(B1,B2)
    Rmat = B1.R_matrix(B2)
    x1 = B1(row1)
    x2 = B2(row2)
    x = T(x1,x2)    
    return(Rmat(x))

def R_matrix_C_two_crystal_rows(n,r1,r2,row1,row2):
    B1 = crystals.KirillovReshetikhin(['C',n,1],r1,1)
    B2 = crystals.KirillovReshetikhin(['C',n,1],r2,1)
    T = crystals.TensorProduct(B1,B2)
    Rmat = B1.R_matrix(B2)
    x1 = B1(row1)
    x2 = B2(row2)
    x = T(x1,x2)    
    return(Rmat(x))

def process_result_R_matrix(t):
    return([t for t in t[0].to_kirillov_reshetikhin_tableau()])

def process_crystal_row(t):
    return([x for x in t.to_kirillov_reshetikhin_tableau()])

def indicator_vector_row(n,row):
    split_row = to_number_matrix(n,[row])
    ans = [0 for j in range(n)]
    bottom = split_row[0]
    for i in range(n):
        if bottom[i] != 0:
            ans[i] = 1
    return(ans)



## TASEP related methods in type C

In [ ]:
def projection_map(n,elt_prod):
    bot_row = elt_prod[0]
    ans = indicator_vector_row(n,bot_row)
    splt = to_number_matrix(n,[bot_row])
    ans = [ans[i]*splt[0][i] for i in range(n)]
    
    for k in range(1,len(lam)):
        x = copy.deepcopy(elt_prod)
        current_row = x[k]
        loc_ind_vec = []
        for i in reversed(range(k)):
            row1 = x[i]
            row2 = current_row
            r1 = len(row1.to_kirillov_reshetikhin_tableau())
            r2 = len(row2.to_kirillov_reshetikhin_tableau())
            
            res = R_matrix_C_two_crystal_rows(n,r1,r2,row1,row2)

            current_row = res[0]

            loc_ind_vec = indicator_vector_row(n,current_row)
            splt = to_number_matrix(n,[res[0]])
            loc_ind_vec = [loc_ind_vec[i]*splt[0][i] for i in range(n)]
        
        ans = [ans[i] + loc_ind_vec[i] for i in range(n)]
            
    return(ans)

def ei_type_C(n,r1,row1,i):
    B1 = crystals.KirillovReshetikhin(['C',n,1],r1,1)
    prow1 = process_row(row1)
    x1 = B1(rows=prow1)
    x = x1.e(i)
    return(x)

def fi_type_C(n,r1,row1,i):
    B1 = crystals.KirillovReshetikhin(['C',n,1],r1,1)
    prow1 = process_row(row1)
    x1 = B1(rows=prow1)
    x = x1.f(i)
    return(x)